# 02 — Deploy as Hosted Agent

Deploy the Impossible Travel investigation workflow as a **Hosted Agent** in Microsoft Foundry.

This notebook packages the **same** workflow built locally in `01-investigation.ipynb`
(ContextEnricher → 10 risk sub-agents → RiskAggregator → PACO) into a container and
deploys it. The `test_cases/*.json` scenarios are bundled into the image so the
deployed agent can answer detections for any UPN found in those files.

### Deployment Lifecycle
1. **Package** — Generate `main.py`, `Dockerfile`, `requirements.txt`, and copy `test_cases/`
2. **Build** — `docker build` the container image (linux/amd64)
3. **Push** — Push to Azure Container Registry
4. **Deploy** — `project.agents.create_version()` registers the image with Foundry
5. **Invoke** — Send incidents via the Responses API

### Platform-Injected Environment Variables
| Variable | Description |
|---|---|
| `FOUNDRY_PROJECT_ENDPOINT` | Foundry project endpoint URL (auto-injected) |
| `FOUNDRY_AGENT_NAME` | Name of the running agent (auto-injected) |
| `APPLICATIONINSIGHTS_CONNECTION_STRING` | App Insights for telemetry (auto-injected) |
| `MODEL_DEPLOYMENT_NAME` | Deployment used by risk sub-agents (set during create_version) |
| `PACO_MODEL_DEPLOYMENT_NAME` | Deployment used by PACO; defaults to `MODEL_DEPLOYMENT_NAME` |

> **Prerequisites**: Run `00-setup.ipynb` first. Docker Desktop must be running.

In [18]:
# Load config

import json
import os
from pathlib import Path

config_path = Path("workshop_config.json")
if not config_path.exists():
    raise FileNotFoundError("workshop_config.json not found. Run 00-setup.ipynb first.")

with open(config_path) as f:
    config = json.load(f)

RESOURCE_GROUP = config["RESOURCE_GROUP"]
ACCOUNT_NAME = config["ACCOUNT_NAME"]
PROJECT_NAME = config["PROJECT_NAME"]
MODEL_DEPLOYMENT_NAME = config["MODEL_DEPLOYMENT_NAME"]
PROJECT_ENDPOINT = config["PROJECT_ENDPOINT"]

from azure.identity import AzureCliCredential
credential = AzureCliCredential()

print(f"✅ Config loaded — deploying to {ACCOUNT_NAME}/{PROJECT_NAME}")

✅ Config loaded — deploying to fndryiac2ttkx3/iac-ops-demo


## Generate Deployment Artifacts

In [19]:
# Sync the shared workflow module + test_cases/ into hosted_agent/ so they
# end up in the Docker build context. The hosted agent's main.py imports
# `investigation_workflow.py` directly — no MAIN_PY string duplication.

import shutil

HOSTED = Path("hosted_agent")
HOSTED.mkdir(exist_ok=True)

# 1) Shared workflow module (single source of truth, mirrored from repo root).
shared_src = Path("investigation_workflow.py")
shared_dst = HOSTED / "investigation_workflow.py"
shutil.copy2(shared_src, shared_dst)
print(f"  ✅ Synced: {shared_dst}")

# 2) Test case scenarios — the hosted agent loads them by UPN at startup.
src_cases = Path("test_cases")
dst_cases = HOSTED / "test_cases"
if dst_cases.exists():
    shutil.rmtree(dst_cases)
dst_cases.mkdir(parents=True, exist_ok=True)
case_count = 0
for case_file in sorted(src_cases.glob("*.json")):
    shutil.copy2(case_file, dst_cases / case_file.name)
    case_count += 1
print(f"  ✅ Bundled {case_count} test case(s) into {dst_cases}/")

# Sanity check that hosted_agent/main.py / Dockerfile / requirements.txt
# (checked-in files) are present in the build context.
for required in ("main.py", "Dockerfile", "requirements.txt"):
    p = HOSTED / required
    if not p.exists():
        raise FileNotFoundError(f"Expected {p} to exist (committed to the repo).")
    print(f"  ✓ Present: {p}")

print("\n📁 Deployment artifacts ready in hosted_agent/")

  ✅ Synced: hosted_agent\investigation_workflow.py
  ✅ Bundled 5 test case(s) into hosted_agent\test_cases/
  ✓ Present: hosted_agent\main.py
  ✓ Present: hosted_agent\Dockerfile
  ✓ Present: hosted_agent\requirements.txt

📁 Deployment artifacts ready in hosted_agent/


## Build, Push & Deploy

In [22]:
import asyncio
import subprocess
import sys

AGENT_NAME = "impossible-travel-investigator"
IMAGE_TAG = "v1"

# Optional manual override — set this if auto-discovery below can't find your ACR
# (e.g. your registry lives in a different resource group / subscription).
#   ACR_LOGIN_SERVER = "myregistry.azurecr.io"
ACR_LOGIN_SERVER = ""

# On Windows, `az` and `docker` are shipped as .cmd shims that CreateProcess
# can't launch directly — use shell=True so cmd.exe resolves them via PATHEXT.
_USE_SHELL = sys.platform == "win32"


def _run_az(args: list[str]) -> str:
    """Run an `az` command and return stdout. Print stderr on failure for diagnostics."""
    result = subprocess.run(
        ["az", *args], capture_output=True, text=True, shell=_USE_SHELL,
    )
    if result.returncode != 0:
        print(f"  ⚠️  az {' '.join(args)} (exit {result.returncode})")
        if result.stderr.strip():
            print(f"     stderr: {result.stderr.strip().splitlines()[0][:200]}")
        return ""
    return result.stdout.strip()


# ── Preflight: Docker daemon must be running ──────────────────
_docker_check = subprocess.run(
    ["docker", "info"], capture_output=True, text=True, shell=_USE_SHELL,
)
if _docker_check.returncode != 0:
    raise RuntimeError(
        "Docker daemon is not reachable. Start Docker Desktop and wait for it to be ready, "
        "then re-run this cell.\n"
        f"  docker info stderr: {_docker_check.stderr.strip().splitlines()[0][:200] if _docker_check.stderr.strip() else '(none)'}"
    )


# ── Step 1: Discover ACR ──────────────────────────────────────
print("▶ Step 1: Discovering Azure Container Registry...")

if not ACR_LOGIN_SERVER:
    # 1a. Some Foundry/AI Services resources expose a linked ACR via properties.containerRegistry.
    acr_id = _run_az([
        "cognitiveservices", "account", "show",
        "--name", ACCOUNT_NAME, "--resource-group", RESOURCE_GROUP,
        "--query", "properties.containerRegistry", "-o", "tsv",
    ])
    if acr_id and acr_id != "None":
        acr_name = acr_id.split("/")[-1] if "/" in acr_id else acr_id
        ACR_LOGIN_SERVER = _run_az([
            "acr", "show", "--name", acr_name, "--query", "loginServer", "-o", "tsv",
        ])
        if ACR_LOGIN_SERVER:
            print(f"  Found ACR linked to Foundry resource: {ACR_LOGIN_SERVER}")

if not ACR_LOGIN_SERVER:
    # 1b. Fall back to the first ACR in the same resource group.
    ACR_LOGIN_SERVER = _run_az([
        "acr", "list", "--resource-group", RESOURCE_GROUP,
        "--query", "[0].loginServer", "-o", "tsv",
    ])
    if ACR_LOGIN_SERVER:
        print(f"  Found ACR in resource group '{RESOURCE_GROUP}': {ACR_LOGIN_SERVER}")

if not ACR_LOGIN_SERVER:
    # 1c. Last resort — scan the entire subscription.
    ACR_LOGIN_SERVER = _run_az([
        "acr", "list", "--query", "[0].loginServer", "-o", "tsv",
    ])
    if ACR_LOGIN_SERVER:
        print(f"  Found ACR in subscription (first match): {ACR_LOGIN_SERVER}")

if not ACR_LOGIN_SERVER:
    raise RuntimeError(
        "Could not discover an Azure Container Registry.\n"
        "Either:\n"
        f"  • Create one:  az acr create -g {RESOURCE_GROUP} -n <name> --sku Basic --admin-enabled true\n"
        "  • Or set ACR_LOGIN_SERVER manually at the top of this cell (e.g. 'myregistry.azurecr.io')."
    )

FULL_IMAGE = f"{ACR_LOGIN_SERVER}/{AGENT_NAME}:{IMAGE_TAG}"
print(f"  ACR:   {ACR_LOGIN_SERVER}")
print(f"  Image: {FULL_IMAGE}")

# ── Step 2: Build ─────────────────────────────────────────────
print("\n▶ Step 2: Building Docker image...")
subprocess.run(
    ["docker", "build", "--platform", "linux/amd64", "-t", f"{AGENT_NAME}:{IMAGE_TAG}", "hosted_agent/"],
    check=True, shell=_USE_SHELL,
)
print("  ✅ Image built")

# ── Step 3: Push ──────────────────────────────────────────────
print("\n▶ Step 3: Pushing to ACR...")
acr_name = ACR_LOGIN_SERVER.split(".")[0]
subprocess.run(["az", "acr", "login", "--name", acr_name], check=True, shell=_USE_SHELL)
subprocess.run(["docker", "tag", f"{AGENT_NAME}:{IMAGE_TAG}", FULL_IMAGE], check=True, shell=_USE_SHELL)
subprocess.run(["docker", "push", FULL_IMAGE], check=True, shell=_USE_SHELL)
print("  ✅ Image pushed")

# ── Step 3.5: Ensure RBAC for image pull + model access ───────
# Foundry's manual deploy path requires three role assignments that the
# `azd` / VS Code Toolkit paths grant automatically. Without these the version
# either fails to pull the image or returns 500 at invoke time.
#
#   (a) AcrPull on the ACR  →  Foundry PROJECT's system-assigned managed identity
#                              (Foundry uses this to pull the image)
#   (b) Cognitive Services OpenAI User on the Foundry account
#                          →  the per-agent runtime identity
#                              (container calls Foundry chat endpoints via this)
#   (c) Foundry User on the Foundry account
#                          →  the per-agent runtime identity
#                              (formerly "Azure AI User"; required at runtime)
print("\n▶ Step 3.5: Ensuring required RBAC role assignments...")

_acr_resource_id = _run_az([
    "acr", "show", "--name", acr_name, "--query", "id", "-o", "tsv",
])
_foundry_account_id = _run_az([
    "cognitiveservices", "account", "show",
    "--name", ACCOUNT_NAME, "--resource-group", RESOURCE_GROUP,
    "--query", "id", "-o", "tsv",
])
_project_principal = _run_az([
    "resource", "show",
    "--ids", f"{_foundry_account_id}/projects/{PROJECT_NAME}",
    "--query", "identity.principalId", "-o", "tsv",
])


def _ensure_role(principal_id: str, role: str, scope: str, label: str) -> None:
    if not principal_id or principal_id == "None":
        print(f"  ⚠️  Skipping '{role}' for {label}: no principal id resolved.")
        return
    existing = _run_az([
        "role", "assignment", "list",
        "--assignee", principal_id, "--scope", scope,
        "--query", f"[?roleDefinitionName=='{role}'] | length(@)", "-o", "tsv",
    ])
    if existing and existing != "0":
        print(f"  ✓ '{role}' already granted to {label} on {scope.split('/')[-1]}")
        return
    print(f"  • Granting '{role}' to {label} ({principal_id[:8]}…) on {scope.split('/')[-1]}")
    _run_az([
        "role", "assignment", "create",
        "--assignee-object-id", principal_id,
        "--assignee-principal-type", "ServicePrincipal",
        "--role", role, "--scope", scope,
    ])


# (a) Image pull
_ensure_role(_project_principal, "AcrPull", _acr_resource_id, "Foundry project MI")

# Determine the per-agent runtime identity (created by Foundry when a version exists).
# On the very first deploy this won't exist yet — we'll re-run after create_version.
try:
    _existing_agent = project_client.agents.get(agent_name=AGENT_NAME).as_dict()
    _runtime_principal = (_existing_agent.get("instance_identity") or {}).get("principal_id")
except Exception:
    _runtime_principal = None

if _runtime_principal:
    # (b) Model API access
    _ensure_role(_runtime_principal, "Cognitive Services OpenAI User", _foundry_account_id, "agent runtime MI")
    # (c) Foundry data-plane access (was "Azure AI User" — rename rolled out 2025)
    _ensure_role(_runtime_principal, "Foundry User", _foundry_account_id, "agent runtime MI")
else:
    print("  ℹ️  Per-agent runtime identity not yet created — will be granted after first create_version.")

# ── Step 4: Create Hosted Agent ───────────────────────────────
print("\n▶ Step 4: Creating Hosted Agent version...")

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import HostedAgentDefinition, ProtocolVersionRecord, AgentProtocol

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True,
)

# Pass the model deployment name to the container. PACO_MODEL_DEPLOYMENT_NAME is
# optional — set it to a different deployment if you want PACO to reason with a
# stronger model than the risk evaluators (matches the pattern in 01-investigation.ipynb).
PACO_MODEL_DEPLOYMENT_NAME = os.environ.get("PACO_MODEL_DEPLOYMENT_NAME", MODEL_DEPLOYMENT_NAME)

agent_version = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="1.0.0")
        ],
        cpu="1", memory="2Gi", image=FULL_IMAGE,
        environment_variables={
            "MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT_NAME,
            "PACO_MODEL_DEPLOYMENT_NAME": PACO_MODEL_DEPLOYMENT_NAME,
        },
    ),
)
print(f"  Agent: {agent_version.name}, Version: {agent_version.version}")

# ── Step 4.5: Grant runtime model-access roles ────────────────
# After the first create_version, Foundry has provisioned the per-agent runtime
# identity. Grant the runtime roles now (idempotent on subsequent deploys).
try:
    _agent_doc = project_client.agents.get(agent_name=AGENT_NAME).as_dict()
    _runtime_principal = (_agent_doc.get("instance_identity") or {}).get("principal_id")
    if _runtime_principal:
        _ensure_role(_runtime_principal, "Cognitive Services OpenAI User", _foundry_account_id, "agent runtime MI")
        _ensure_role(_runtime_principal, "Foundry User", _foundry_account_id, "agent runtime MI")
        print("  ℹ️  Runtime RBAC propagation can take 30–90s before the agent can call models.")
except Exception as _exc:
    print(f"  ⚠️  Could not resolve runtime identity after create_version: {_exc}")

# ── Step 5: Poll for active ───────────────────────────────────
print("\n▶ Step 5: Waiting for agent to become active...")
while True:
    version_info = project_client.agents.get_version(
        agent_name=AGENT_NAME, agent_version=agent_version.version,
    )
    status = version_info["status"]
    print(f"  Status: {status}")
    if status == "active":
        print("  ✅ Agent is ready!")
        break
    elif status == "failed":
        print(f"  ❌ Provisioning failed: {version_info.get('error')}")
        break
    await asyncio.sleep(5)

▶ Step 1: Discovering Azure Container Registry...
  Found ACR in subscription (first match): htamlcr.azurecr.io
  ACR:   htamlcr.azurecr.io
  Image: htamlcr.azurecr.io/impossible-travel-investigator:v1

▶ Step 2: Building Docker image...
  ✅ Image built

▶ Step 3: Pushing to ACR...
  ✅ Image pushed

▶ Step 3.5: Ensuring required RBAC role assignments...
  ✓ 'AcrPull' already granted to Foundry project MI on htamlcr
  ✓ 'Cognitive Services OpenAI User' already granted to agent runtime MI on fndryiac2ttkx3
  ✓ 'Foundry User' already granted to agent runtime MI on fndryiac2ttkx3

▶ Step 4: Creating Hosted Agent version...
  Agent: impossible-travel-investigator, Version: 8
  ✓ 'Cognitive Services OpenAI User' already granted to agent runtime MI on fndryiac2ttkx3
  ✓ 'Foundry User' already granted to agent runtime MI on fndryiac2ttkx3
  ℹ️  Runtime RBAC propagation can take 30–90s before the agent can call models.

▶ Step 5: Waiting for agent to become active...
  Status: creating
  Status

## Invoke the Deployed Agent

In [23]:
from pydantic import BaseModel, Field
from typing import Any

# Re-define the InvestigationVerdict shape (matches 01-investigation.ipynb)
# so we can parse and pretty-print the deployed agent's structured response.
class ActionItem(BaseModel):
    action: str
    target: str = ""
    value: str = ""
    riskFactor: str = ""
    destructive: bool = False
    requiresApproval: bool = False

class InvestigationVerdict(BaseModel):
    verdict: str
    confidence: str
    reasoning: str
    actionPlan: list[ActionItem] = Field(default_factory=list)
    narrative: str = ""


# Load the same test case the local notebook uses (must be one of the JSON
# scenarios bundled into the container under hosted_agent/test_cases/).
# Change TEST_CASE to exercise a different scenario.
from pathlib import Path

TEST_CASE = "03_mfa_fatigue_attack"
case_path = Path("test_cases") / f"{TEST_CASE}.json"
with open(case_path, encoding="utf-8") as f:
    test_case_data = json.load(f)

detection_payload = test_case_data["detection"]
incident_json = json.dumps(detection_payload)

print(f"▶ Invoking deployed agent with scenario: {test_case_data['name']}")
print(f"   User:      {detection_payload['PrimaryEntityId']}")
print(f"   Expected:  {test_case_data.get('expected_verdict', '—')}"
      f" ({test_case_data.get('expected_confidence', '—')} confidence)")

openai_client = project_client.get_openai_client(agent_name=AGENT_NAME)
response = openai_client.responses.create(input=incident_json)

print("\n" + "═" * 70)
print("📋 HOSTED AGENT RESPONSE")
print("═" * 70)
print(response.output_text)

try:
    hosted_verdict = InvestigationVerdict.model_validate_json(response.output_text)
    print(f"\n  Verdict:    {hosted_verdict.verdict}")
    print(f"  Confidence: {hosted_verdict.confidence}")
    print(f"  Actions:    {len(hosted_verdict.actionPlan)}")
except Exception:
    print("  (Could not parse structured verdict from response)")

▶ Invoking deployed agent with scenario: MFA Fatigue Attack
   User:      bob.chen@contoso.com
   Expected:  AccountCompromised (High confidence)

══════════════════════════════════════════════════════════════════════
📋 HOSTED AGENT RESPONSE
══════════════════════════════════════════════════════════════════════
{
  "verdict": "AccountCompromised",
  "confidence": "High",
  "reasoning": "The investigation revealed multiple strong indicators of account compromise on user bob.chen@contoso.com. High evidence scores from LocationAnomaly (85), TokenAnomaly (85), BruteForcePattern (85), and PostLoginSuspiciousActivity (85) establish a clear pattern of malicious activity. The detection of impossible travel between Seattle and a suspicious proxy IP in Sofia, multiple rapid failed login attempts followed by success, concurrent sessions with unusual app activity on Microsoft Graph and SharePoint Online, and changes to MFA methods immediately after suspicious logins collectively confirm a high lik

In [ ]:
# Cleanup (Optional)
# Uncomment the lines below to delete the hosted agent.

# print("▶ Cleaning up hosted agent...")
# project_client.agents.delete(agent_name=AGENT_NAME)
# print("✅ Agent deleted")

## Architecture & Production Notes

### SOC Document → Agent Framework Mapping

| SOC Document Concept | Agent Framework Implementation |
|---|---|
| Sentinel Analytics Rule | `NormalizedDetection` — input trigger |
| Context Agent | `ContextEnricher(Executor)` — deterministic, no LLM |
| Identity Agent (10 tools) | 10 specialized `Agent` instances with fan-out |
| RiskEvidence generation | Structured output via `response_format=RiskEvidence` |
| ThreatDecisionRecord | `RiskAggregator(Executor)` — deterministic assembly |
| ThreatDecisionPolicy | Embedded in PACO's system prompt |
| PACO decision maker | `Agent` with `response_format=InvestigationVerdict` |
| ActionPlan generation | Part of PACO's structured output |
| Pipeline orchestration | `WorkflowBuilder` with fan-out/fan-in graph |

### Production Considerations

1. **Real Data Sources via MCP**: Replace `@tool` simulated data with MCP connections to:
   - Microsoft Sentinel (KQL queries for sign-in logs, audit logs, alerts)
   - Microsoft Entra ID (user profiles, roles, groups)
   - Microsoft Defender XDR (incident correlation)

2. **Human-in-the-Loop**: Use `ctx.request_info()` in the workflow
   to pause and await analyst approval before executing destructive actions on privileged users.

3. **Tracing & Observability**: Hosted Agents auto-inject Application Insights.
   View investigation traces in Foundry Portal → Tracing.

4. **Scaling**: Each Sentinel detection triggers one workflow run.
   Foundry Hosted Agents scale-to-zero with 15-minute idle timeout.